# Exercise 2.5: Merging & Combining Datasets (Angola IEA and INE trade)

Three separate join problems, on purpose:

- **Part A** attaches province names to the survey with a lookup table, using a
  deliberately outdated lookup so the audit tools have something to find.
- **Part B** appends the Q3 2025 wave of the same survey to the Q4 wave, where
  the two files share only 20 of their 260 and 206 columns.
- **Part C** merges Angola's export and import tables to compute a trade balance.

Part C uses a different dataset. There is no sensible join between an individual
level labour survey and country level trade totals, and pretending otherwise
would teach a bad habit.

> **Pipeline:** run Exercise 2.4 first. Writes three files to `20_processed/`.

### Path Setup (run first)

In [ ]:
import os

import numpy as np
import pandas as pd

DATA_PROC_DIR = '../../data/20_processed'
DATA_SURVEY_DIR = '../../data/0_raw/angola/employment_survey'
DATA_TRADE_DIR = '../../data/0_raw/angola/international_trade'

features_path = os.path.join(DATA_PROC_DIR, 'angola_iea_2025q4_features.csv')

STR_COLS = {
    'household_id': 'string', 'person_no': 'string',
    'cluster_id': 'string', 'province_code': 'string',
}
df = pd.read_csv(features_path, dtype=STR_COLS)

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
print('Survey:', df.shape)
df[['household_id', 'province_code', 'age', 'lf_status_strict']].head()

---

# Part A: attaching province names with a lookup

## Task 1: A lookup table, as published before 2024

Angola reorganised its provinces in 2024, going from 18 to 21. Icolo e Bengo,
Moxico Leste and Cuando are new. The lookup below is the **old** 18 province
list, which is exactly what you get if you copy a reference table from an older
publication.

In [ ]:
province_lookup_old = pd.DataFrame({
    'province_code': ['10', '11', '12', '13', '14', '15', '16', '17', '18',
                      '19', '20', '21', '22', '23', '24', '25', '26', '27'],
    'province_name_ref': ['Cabinda', 'Zaire', 'Uíge', 'Bengo', 'Luanda',
                          'Cuanza-Norte', 'Cuanza-Sul', 'Malanje', 'Lunda-Norte',
                          'Lunda-Sul', 'Moxico', 'Bié', 'Huambo', 'Benguela',
                          'Namibe', 'Huila', 'Cunene', 'Cubango'],
})
print('Lookup rows:', len(province_lookup_old))
province_lookup_old.head()

## Task 2: Join key hygiene, before you merge

Most failed merges are a key that is text on one side and a number on the other,
or a stray space. Check dtypes, whitespace and missing keys on both sides first.

In [ ]:
print('Survey key dtype:', df['province_code'].dtype)
print('Lookup key dtype:', province_lookup_old['province_code'].dtype)
print()
print('Missing keys, survey:', df['province_code'].isna().sum())
print('Missing keys, lookup:', province_lookup_old['province_code'].isna().sum())
print('Duplicate keys, lookup:', province_lookup_old['province_code'].duplicated().sum())

**Questions:**

- What dtype is the join key on each side, and why does it matter that they
  match?
- Are there any missing keys on either side, or duplicate keys in the lookup?
  What does that imply for the row count after a left join?

## Task 3: Merge, then audit with `indicator`

A left join keeps every survey row. `indicator=True` adds a `_merge` column
labelling each row `both`, `left_only` or `right_only`, which is how you find out
what silently failed to match.

In [ ]:
merged = pd.merge(df, province_lookup_old, on='province_code',
                  how=  # your code here , indicator=  # your code here )

print('Rows:', len(df), '->', len(merged))
print(merged['_merge'].value_counts())

In [ ]:
unmatched = merged[merged['_merge'] ==   # your code here ]
print('Unmatched rows:', len(unmatched),
      f'({len(unmatched) / len(merged) * 100:.1f}%)')
print()
print(unmatched['province_code'].value_counts().sort_index())

In [ ]:
inner = pd.merge(df, province_lookup_old, on='province_code', how='inner')
outer = pd.merge(df, province_lookup_old, on='province_code', how='outer')
print('inner:', len(inner), '| left:', len(merged), '| outer:', len(outer))

**Questions:**

- Does the row count change after the left join? What must be true of the
  right hand key for that to hold?
- How many rows are `left_only`, and which province codes do they belong to?
  Why do those codes fail to match?
- What happens to those rows under an inner join instead, and why is that
  worse than a visible `NaN`?
- Which of `province_name` (from 2.4) and `province_name_ref` (from this
  lookup) is correct, and how would you prove it?

## Task 4: Cardinality, and making the assumption explicit

If the right hand key is not unique, every duplicate match multiplies rows.
`validate=` states your assumption and raises instead of silently inflating.

In [ ]:
# A lookup that accidentally lists Cabinda twice
bad_lookup = pd.concat([province_lookup_old, province_lookup_old.head(1)],
                       ignore_index=True)

exploded = pd.merge(df, bad_lookup, on='province_code', how='left')
print('Rows before:', len(df), '-> after the bad merge:', len(exploded))
print('Extra rows:', len(exploded) - len(df))

In [ ]:
try:
    pd.merge(df, bad_lookup, on='province_code', how='left',
             validate=  # your code here )
except Exception as error:
    print(type(error).__name__, '->', error)

**Questions:**

- What happens to the row count when the lookup has a duplicate key, and why?
- What does `validate='many_to_one'` do when the assumption is violated?
- What is the difference between `one_to_one`, `one_to_many` and
  `many_to_one`, and why does stating the assumption matter?

## Task 5: Post merge validation, then save

A merge is finished when you have confirmed the result, not when the code ran.

In [ ]:
final = pd.merge(df, province_lookup_old, on='province_code', how='left')

print('Row count:', len(df), '->', len(final))
print('Duplicate person keys:', final.  # your code here )
print('Unmatched province rate:', round(final['province_name_ref'].isna().mean(), 4))

In [ ]:
final = final.reset_index(drop=True)
out_path = os.path.join(DATA_PROC_DIR, 'angola_iea_2025q4_analysis.csv')
final.to_csv(out_path, index=False)
print('Saved:', out_path, '|', final.shape)

**Questions:**

- Does the row count match before and after the final merge? Are there any
  duplicate person keys?
- What is the unmatched province rate, and is it one you can explain?

---

# Part B: appending the Q3 and Q4 waves

## Task 6: Load the previous quarter

`IEA_III_TRIMESTRE_2025.sav` is the same survey, one quarter earlier. It uses the
questionnaire's own variable names rather than the ILO mnemonics of the Q4 file,
so almost nothing lines up.

In [ ]:
q3_path = os.path.join(DATA_SURVEY_DIR, 'IEA_III_TRIMESTRE_2025.sav')
q3_full = pd.read_spss(q3_path, convert_categoricals=False)

print('Q3:', q3_full.shape)
print('Q4 features:', df.shape)
print('Column names in common:', len(set(q3_full.columns) & set(df.columns)))

## Task 7: What a naive `concat` does

`pd.concat` aligns on column names and fills every gap with `NaN`, without a
single warning. Try it and measure the damage.

In [ ]:
naive = pd.concat(  # your code here: q3_full and df, ignore_index=True )

print('Naive concat:', naive.shape)
mostly_empty = (naive.isna().mean() > 0.99).sum()
print(f'Columns more than 99% empty: {mostly_empty} of {naive.shape[1]}')
naive.iloc[:3, :6]

**Questions:**

- What is the shape of the naive concat, and how many columns are more than
  99% empty?
- Does `pd.concat` warn you when this happens? Whose mistake is it?
- Why is this kind of failure more dangerous than an error?

## Task 8: Do it properly, by harmonising first

Pick the variables that exist in both waves, rename the Q3 ones to the Q4 names,
confirm the two frames have identical columns, then stack them with a `wave`
column so no row loses its origin.

In [ ]:
Q3_RENAME = {
    'NIDF': 'household_id', 'PROV': 'province_code', 'AREA_RESID': 'area_type',
    'S02_01': 'sex', 'S02_02': 'age', 'S4_01': 'worked_for_pay',
    'S4_02': 'worked_own_account', 'S4_03': 'worked_family_business',
    'S4_09': 'absent_from_job', 'S8_01': 'sought_work', 'S8_12': 'available_now',
    'POND_IEA_III_TRIM_2025_IND': 'weight_ind',
}

q3 = q3_full[list(Q3_RENAME)].  # your code here: rename with Q3_RENAME
q3['household_id'] = q3['household_id'].astype('int64').astype('string')
q3['province_code'] = q3['province_code'].astype('int64').astype('string').str.zfill(2)

print('Q3 harmonised:', q3.shape)
q3.head()

In [ ]:
SHARED = [
    'household_id', 'province_code', 'area_type', 'sex', 'age',
    'worked_for_pay', 'worked_own_account', 'worked_family_business',
    'absent_from_job', 'sought_work', 'available_now', 'weight_ind',
]

q3_slim = q3[SHARED].  # your code here: assign wave='2025Q3'
q4_slim = df[SHARED].assign(wave='2025Q4')

print('Columns identical:', list(q3_slim.columns) == list(q4_slim.columns))

waves = pd.concat([q3_slim, q4_slim], ignore_index=True)
print('Stacked:', waves.shape)
print('Any column entirely empty:', waves.isna().all().any())
print(waves['wave'].value_counts())

**Questions:**

- What is the shape of the harmonised stack, and is any column entirely empty?
- Why must the `wave` column be added before stacking rather than after?
- How many shared variables are there out of the 206 and 260 total columns in
  each wave? What does that imply about harmonising across waves?

## Task 9: Cross wave sanity checks

Two independent samples of the same population should agree on the things that do
not change quickly. If they do not, the append is wrong.

In [ ]:
population = waves.groupby('wave')['weight_ind'].  # your code here
print('Weighted population by wave:')
print(population.round(0))
difference = abs(population.iloc[0] - population.iloc[1]) / population.iloc[1] * 100
print(f'Relative difference: {difference:.2f}%')

In [ ]:
# The harmonised definition can only use the variables both waves carry
for wave in ['2025Q3', '2025Q4']:
    sample = waves[waves['wave'] == wave]
    working_age = sample['age'] >= 15
    employed = working_age & (
        (sample['worked_for_pay'] == 1)
        | (sample['worked_own_account'] == 1)
        | (sample['absent_from_job'] == 1)
    )
    unemployed = (working_age & ~employed
                  & (sample['sought_work'] == 1)
                  & (sample['available_now'] == 1))
    weights = sample['weight_ind']
    rate = weights[unemployed].sum() / weights[employed | unemployed].sum() * 100
    print(f'{wave} harmonised strict unemployment: {rate:.1f}%')

In [ ]:
out_path = os.path.join(DATA_PROC_DIR, 'angola_iea_waves_q3_q4.csv')
waves.to_csv(out_path, index=False)
print('Saved:', out_path, '|', waves.shape)

**Questions:**

- What are the weighted populations for each wave, and how far apart are
  they? What does that agreement tell you?
- What is the harmonised strict unemployment rate in each wave?
- Why does the harmonised Q4 rate differ from the 14.5% computed in 2.4? Is
  that a bug?
- If the two populations had differed by 30% instead, what would you suspect?

---

# Part C: Angola's trade balance

Different dataset, different join. `Comercio Externo de Bens por Países
Parceiros.xlsx` is published by INE with four sheets: exports and imports, each
in kwanzas and in US dollars.

The sheets are formatted for human readers, so loading them takes work: two title
rows above the header, a blank row, a `Total Geral` row, and a source footer at
the bottom.

In [ ]:
trade_path = os.path.join(DATA_TRADE_DIR,
                          'Comercio Externo de Bens por Países Parceiros.xlsx')

print(pd.ExcelFile(trade_path).sheet_names)

In [ ]:
# What the raw sheet looks like before any cleaning
pd.read_excel(trade_path, sheet_name='Exportação por Países (USD)',
              header=None, nrows=6).iloc[:, :5]

## Task 10: Load a sheet properly

`skiprows=2` puts the real header row in place. The country code must be read as
text, the header names carry an embedded newline, and the total and footer rows
both lack a country name, which makes them easy to remove together.

In [ ]:
def load_trade_sheet(path, sheet):
    """Load one INE trade sheet and strip its title, total and footer rows."""
    frame = pd.read_excel(path, sheet_name=sheet,
                          # your code here: skiprows and dtype
                          )
    frame.columns = frame.columns.str.replace('\n', ' ', regex=False).str.strip()
    frame = frame[  # your code here: rows where País is not null ].copy()
    return frame.rename(columns={'Código': 'country_code', 'País': 'country_name'})


exports = load_trade_sheet(trade_path, 'Exportação por Países (USD)')
imports = load_trade_sheet(trade_path, 'Importação por Países (USD)')

print('Exports:', exports.shape, '| Imports:', imports.shape)
print('Columns:', list(exports.columns)[:4], '...', list(exports.columns)[-2:])
exports.head()

**Questions:**

- How many rows does each sheet give, and what does the extra row beyond the
  248 countries represent?
- Why does filtering on `frame['País'].notna()` remove the blank row, the
  total row and the footer in one step? Why is that safer than dropping rows
  by position?
- Why does `dtype={'Código': str}` matter?
- Why would the embedded newline in a header like `Ano\n2004` break later
  code if it were not removed?

## Task 11: Merge exports against imports

Both tables have one row per country, so this is a one to one merge. An outer
join keeps partners that appear on only one side.

In [ ]:
YEAR = 'Ano 2025'

trade = pd.merge(
    exports[['country_code', 'country_name', YEAR]].rename(columns={YEAR: 'exports_usd'}),
    imports[['country_code', YEAR]].rename(columns={YEAR: 'imports_usd'}),
    on='country_code',
    how=  # your code here,
    indicator=  # your code here,
    validate=  # your code here,
)
print('Merged:', trade.shape)
print(trade['_merge'].value_counts())

In [ ]:
trade['balance_usd'] =   # your code here: exports minus imports, treating gaps as 0

print('Largest surpluses:')
print(trade.nlargest(5, 'balance_usd')[['country_name', 'exports_usd',
                                        'imports_usd', 'balance_usd']].to_string(index=False))
print()
print('Largest deficits:')
print(trade.nsmallest(5, 'balance_usd')[['country_name', 'exports_usd',
                                         'imports_usd', 'balance_usd']].to_string(index=False))

In [ ]:
print(trade[trade['country_code'] == 'ZZ'][
    ['country_code', 'country_name', 'exports_usd', 'imports_usd', 'balance_usd']])

**Questions:**

- Are all 249 countries `both`, and does `validate='one_to_one'` pass? What
  does that confirm?
- Which country is the largest surplus partner, and which is the largest
  deficit partner? Does that match Angola's trade profile?
- What does the `ZZ`, Desconhecido, row represent, and why should it not be
  silently dropped?
- Why is `fillna(0)` before subtracting a decision rather than a formality?

## Task 12: Save the trade table

In [ ]:
trade = trade.drop(columns='_merge').reset_index(drop=True)
out_path = os.path.join(DATA_PROC_DIR, 'angola_trade_partners.csv')
trade.  # your code here: to_csv with index=False
print('Saved:', out_path, '|', trade.shape)

**Questions:**

- How many files does this notebook write in total, and what does each one
  contain?
- What is the difference between merging and appending, and which parts of
  this notebook did each?